# 01 — Single-Domain LoRA Baselines (`B1` Target-Only & `B2` Source-Only)

Trains `B1` (TextVQA-only) and `B2` (DocVQA-only), syncing checkpoints to private HF Hub (`gamusa/crossvqa-checkpoints`) and personal Google Drive.


In [ ]:
# ==============================================================================
# 1. COLAB SETUP, REPO SYNC & GPU CHECK (L4 > A100 > G4 > T4 FALLBACK)
# ==============================================================================
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not Path("TRAIN/src").exists():
    if Path("VQA_Final_Thesis/TRAIN/src").exists():
        os.chdir("VQA_Final_Thesis")
    else:
        subprocess.run(["git", "clone", "https://github.com/NhanChanDeo/VQA_Final_Thesis.git"], check=True)
        os.chdir("VQA_Final_Thesis")

subprocess.run(["git", "pull", "origin", "main"], check=True)
print("Working directory:", Path.cwd())

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.49.0", "peft", "bitsandbytes", "accelerate",
        "datasets", "polars", "pyyaml", "safetensors", "huggingface_hub",
        "hf_transfer", "umap-learn", "scikit-learn", "qwen-vl-utils",
    ],
    check=True,
)

import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No CUDA GPU detected. Switch Colab runtime to L4, A100, G4, or T4.")

In [ ]:
# ==============================================================================
# 2. MASKED API KEY INPUT BOX & GOOGLE DRIVE CHECKPOINT BACKUP
# ==============================================================================
import os
from getpass import getpass
from pathlib import Path
from huggingface_hub import HfApi

# 2A. Interactive masked HF WRITE token (never stored in notebook source/outputs)
HF_TOKEN = os.environ.get("HF_TOKEN") or None
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    print("Enter your Hugging Face WRITE Token (input is masked and never saved):")
    HF_TOKEN = getpass("HF Write Token: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
assert HF_TOKEN, "HF_TOKEN is required to save/restore private checkpoints."

api = HfApi(token=HF_TOKEN)
user_info = api.whoami()
print(f"Authenticated Hugging Face account: {user_info['name']}")

# 2B. Mount Google Drive for dual-redundant checkpoint backup
GDRIVE_BACKUP_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_ckpt_root = Path("/content/drive/MyDrive/CrossVQA_Checkpoints/ALT_THREE_SPLITS")
    drive_ckpt_root.mkdir(parents=True, exist_ok=True)
    GDRIVE_BACKUP_DIR = str(drive_ckpt_root)
    print(f"Google Drive backup active: {GDRIVE_BACKUP_DIR}")
except Exception as e:
    print(f"Google Drive mount skipped ({e}); continuing with private HF Hub checkpoints only.")

In [ ]:
# ==============================================================================
# 3. FAST NVMe SNAPSHOT OF `three-splits` SHARDS TO /content/data/
# ==============================================================================
from pathlib import Path
from huggingface_hub import snapshot_download

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

for ds_name, repo_id in [
    ("docvqa", "gamusa/docvqa-canonical"),
    ("textvqa", "gamusa/textvqa-canonical"),
]:
    target = LOCAL_DATA_DIR / ds_name
    print(f"Syncing {repo_id}@three-splits -> {target} ...")
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision="three-splits",
        allow_patterns=["data/*.parquet"],
        local_dir=str(target),
        token=os.environ.get("HF_TOKEN"),
    )
print("Snapshot ready on ephemeral NVMe scratch.")

In [ ]:
# ==============================================================================
# 4. RUN B1 (TEXTVQA-ONLY) OR B2 (DOCVQA-ONLY) BASELINE
# ==============================================================================
import subprocess
import sys

EXP = "B1"                       # "B1" (TextVQA) or "B2" (DocVQA)
MODEL_KEY = "qwen2_vl_2b"        # "qwen2_vl_2b" | "smolvlm_base" | "smolvlm_500m"
HARDWARE_PROFILE = "auto"        # Options: "auto", "l4_24gb", "a100_40gb", "g4_96gb", "t4_16gb"
SEED = 42                        # Seeds: 42, 123, 999
INCLUDE_OCR = False              # False = Track A (OCR-free primary); True = Track B ablation
MAX_TRAIN_STEPS = None           # Set e.g. 20 for smoke test
MAX_EVAL_SAMPLES = None

cmd = [
    sys.executable, "TRAIN/ALT_THREE_SPLITS/scripts/run_baseline.py",
    "--exp", EXP,
    "--model", MODEL_KEY,
    "--seed", str(SEED),
    "--hardware_profile", HARDWARE_PROFILE,
    "--local_data_dir", str(LOCAL_DATA_DIR),
    "--hf_ckpt_repo", "gamusa/crossvqa-checkpoints",
    "--resume_from_hub",
]
if GDRIVE_BACKUP_DIR:
    cmd.extend(["--gdrive_backup_dir", GDRIVE_BACKUP_DIR])
if INCLUDE_OCR:
    cmd.append("--include_ocr")
if MAX_TRAIN_STEPS is not None:
    cmd.extend(["--max_train_steps", str(MAX_TRAIN_STEPS)])
if MAX_EVAL_SAMPLES is not None:
    cmd.extend(["--max_eval_samples", str(MAX_EVAL_SAMPLES)])

subprocess.run(cmd, check=True)

In [ ]:
# ==============================================================================
# 5. INSPECT RESULTS & SYNC LIGHTWEIGHT LOGS TO GITHUB (WEIGHTS EXCLUDED)
# ==============================================================================
import json
import subprocess
from getpass import getpass
from pathlib import Path

# Preview generated metrics/run cards
for p in sorted(Path("TRAIN/ALT_THREE_SPLITS/runs").rglob("val_metrics.json")):
    print("===", p, "===")
    print(p.read_text(encoding="utf-8")[:600])

# Optional masked GitHub push for lightweight JSON/YAML/PNG run artifacts
PUSH_LOGS_TO_GITHUB = False  # Set True when ready to commit logs from Colab
if PUSH_LOGS_TO_GITHUB:
    subprocess.run(["git", "add", "TRAIN/ALT_THREE_SPLITS/runs/"], check=True)
    subprocess.run(["git", "commit", "-m", "chore(runs): sync Colab training metrics and logs"], check=False)
    gh_token = os.environ.get("GH_TOKEN") or getpass("GitHub PAT (masked, optional): ").strip()
    if gh_token:
        remote_url = f"https://{gh_token}@github.com/NhanChanDeo/VQA_Final_Thesis.git"
        subprocess.run(["git", "push", remote_url, "HEAD:main"], check=True)
        print("Synced lightweight logs to origin/main.")